# LiDAR Ground Slope & Curvature Estimation

Rebuilt pipeline, inspired by Santo et al. (2025) *"Ground Segmentation for LiDAR Point Clouds ... Using a Hybrid Neural–Geometric Approach"*. We ignore their neural component (which they use only for sparse regions) and focus on the geometric core.

### The pipeline — an analogy

Imagine you're a cartographer standing in a park full of trees, benches and people, and your job is to map how hilly the **ground** is.

1. **Stage 1 — Remove the clutter.** This is done with Patchwork++
2. **Stage 2 — Measure the terrain.** With the clutter gone, lay a grid over the floor. In each cell, fit a tiny plane with PCA. The plane's **tilt** is the local slope; how well the points fit that plane is the local **curvature** (flat asphalt ≈ 0, rolling hills > 0).
3. **Stage 3 — Stitch frames together.** Run Stages 1–2 on each LiDAR frame along a drive. Record slope/curvature at the sensor's position per frame. Plot the trajectory coloured by what the terrain did — ready to line up against an IMU trace.

### Why this instead of "bottom 20% + RANSAC"?

A single RANSAC plane over the lowest 20% assumes the world is one flat surface. It breaks on hills, ramps, and parking-lot speed bumps — everything gets averaged into one tilted pancake. The CZM/PCA approach handles local geometry per sector, so a road that slopes up into a hill is still correctly classified as ground everywhere along it.

### Why PCA (not RANSAC) for the local planes?

PCA is deterministic, fast, and uses every point in a sector (so it's robust to density variations). RANSAC is stochastic and spends most of its effort guessing which 3 points to start from. Once we already know the points in a sector are *probably* ground (after Stage 1), PCA is the cleaner tool. You can still tune distance thresholds — they just appear as the `k_sigma` and `absolute_max_dist` parameters below. (The paper does the same — see Section 3.2 / Equations 2–3.)

In [ ]:
# Run once if you don't have these:
# !pip install numpy matplotlib pypatchworkpp
# !pip install rosbags          # only needed for MCAP support

import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import pypatchworkpp

np.set_printoptions(suppress=True, precision=4)


# ==========================================================
# WORKFLOW SELECTION
# ----------------------------------------------------------
# Choose where the input point cloud comes from:
#   "bin"  -> a nuScenes-format .bin file (current default)
#   "mcap" -> a frame from a ROS bag (.mcap file)
#
# Everything downstream of cell 3 is identical -- the
# unified loader returns an (N, 3) XYZ array either way.
# ==========================================================
WORKFLOW = "mcap"          # "bin"  or  "mcap"

# Where to save the ground points after
# Patchwork++ as a nuScenes-format .bin file. Set
# EXPORT_GROUND = False to disable.
EXPORT_GROUND   = False
GROUND_BIN_PATH = r"C:\Users\JRepa\OneDrive - Delft University of Technology\Documenten\02. TU Delft\2025-2026\BEP\Data\Ground points\ground_points.bin"


# ==========================================================
# How to select a frame from the MCAP file:
#   Option A: by index number (0 = first frame)
#   Option B: by timestamp (paste from Foxglove)
# Fill in ONE of these and set the other to None.
# ==========================================================
MCAP_FRAME_INDEX     = None                # e.g. 0, 100, 500
MCAP_FRAME_TIMESTAMP = 1762960983.861245797      # paste from Foxglove
MCAP_TOPIC           = None      # This is the default if only one lidar topic is present. Otherwise, set to the topic name you want.
# ==========================================================
# SINGLE SOURCE OF TRUTH FOR SENSOR-SPECIFIC CONFIGURATION
# ----------------------------------------------------------
# Change these when you move from the nuScenes dataset to
# your own LiDAR test rig. Nothing else in the notebook needs
# to be touched -- the Patchwork++ object is rebuilt from
# this dict whenever you re-run cell 8.
# ==========================================================
SENSOR_CONFIG = {
    # Height of the LiDAR above the ground plane [m].
    # nuScenes default: ~1.84 m.  SemanticKITTI: ~1.70 m.
    # For your own setup: measure from the ground to the
    # optical centre of the sensor and put the number here.
    "sensor_height": 1.84,

    # Sensor range limits [m] -- crop points outside this band.
    # Increase max_range for long-range sensors (e.g. OS1-128
    # can see beyond 100 m) or for off-road data.
    "min_range":  1.0,
    "max_range": 50.0,

    # If True, prints Patchwork++'s internal status to stdout
    # on every frame. Useful when tuning, noisy in production.
    "verbose": False,
}


## 1. Load a single point cloud frame

Two formats are supported:

* **`.bin`** -- nuScenes layout: `float32` array reshaped to `(N, 5)` -> `[x, y, z, intensity, ring_index]`. We only need XYZ.
* **`.mcap`** -- a ROS bag containing one or more `sensor_msgs/PointCloud2` topics. Each frame is one message; we deserialise it with the [`rosbags`](https://gitlab.com/ternaris/rosbags) library and extract the XYZ fields.

Switch between them by setting `WORKFLOW` in cell 1. Both paths produce the same `(N, 3)` `float64` array, so the rest of the notebook is unchanged.


In [ ]:
# ------------------------------------------------------------
# Loader 1: nuScenes-format .bin file
# ------------------------------------------------------------
def load_nuscenes_bin(path):
    """Load (N, 3) XYZ from a nuScenes-format .bin file."""
    pts = np.fromfile(path, dtype=np.float32).reshape(-1, 5)
    return pts[:, :3].astype(np.float64)  # float64 for numerical stability in PCA


# ------------------------------------------------------------
# Loader 2: a single frame from a ROS .mcap bag
# ------------------------------------------------------------
# PointField.datatype -> (numpy dtype string, byte size)
_PF_DTYPE = {1: ("i1", 1), 2: ("u1", 1), 3: ("i2", 2), 4: ("u2", 2),
             5: ("i4", 4), 6: ("u4", 4), 7: ("f4", 4), 8: ("f8", 8)}


def _pointcloud2_to_xyz(msg):
    """Extract XYZ as (N, 3) float64 from a deserialised PointCloud2 message.

    Handles arbitrary field layouts (xyz, xyzi, xyzir, etc.) and skips
    any rows containing NaN/inf -- which are common in PointCloud2 and
    represent invalid returns.
    """
    field_map = {f.name: f for f in msg.fields}
    missing = [k for k in ("x", "y", "z") if k not in field_map]
    if missing:
        raise RuntimeError(f"PointCloud2 missing field(s) {missing}. "
                           f"Available: {list(field_map.keys())}")

    n = msg.width * msg.height
    raw = np.frombuffer(msg.data, dtype=np.uint8).reshape(n, msg.point_step)

    xyz = np.empty((n, 3), dtype=np.float64)
    for i, name in enumerate(("x", "y", "z")):
        f = field_map[name]
        dtype_str, size = _PF_DTYPE[f.datatype]
        col_bytes = raw[:, f.offset:f.offset + size].copy()  # contiguous
        xyz[:, i] = col_bytes.view(np.dtype(dtype_str)).flatten().astype(np.float64)

    # Drop invalid points (PointCloud2 commonly contains NaN/inf for missed returns)
    valid = np.isfinite(xyz).all(axis=1)
    return xyz[valid]


def load_mcap_frame(path, topic=None, frame_index=0, list_topics=False):
    """Load (N, 3) XYZ from a single PointCloud2 message inside an MCAP bag.

    Requires `pip install rosbags`. Works for both ROS1 and ROS2 MCAP files.

    Parameters
    ----------
    path : str
        Path to the .mcap file.
    topic : str or None
        ROS topic to extract. If None, the first PointCloud2 topic is used.
    frame_index : int
        Which message on that topic to load (0 = first message in the bag).
    list_topics : bool
        If True, print the available PointCloud2 topics and return None.
        Useful for inspection on a brand-new bag.
    """
    from rosbags.highlevel import AnyReader
    from pathlib import Path

    with AnyReader([Path(path)]) as reader:
        pc2_conns = [c for c in reader.connections
                     if c.msgtype == "sensor_msgs/msg/PointCloud2"]
        if not pc2_conns:
            raise RuntimeError("No PointCloud2 topics found in this MCAP.")

        if list_topics:
            print("PointCloud2 topics in this bag:")
            for c in pc2_conns:
                print(f"  {c.topic}  ({c.msgcount} messages)")
            return None

        if topic is None:
            chosen = pc2_conns[0]
            print(f"[mcap] Auto-selected topic: {chosen.topic} "
                  f"({chosen.msgcount} messages available)")
        else:
            chosen = next((c for c in pc2_conns if c.topic == topic), None)
            if chosen is None:
                avail = [c.topic for c in pc2_conns]
                raise RuntimeError(f"Topic '{topic}' not found. Available: {avail}")

        seen = 0
        for conn, ts, raw in reader.messages(connections=[chosen]):
            if seen == frame_index:
                msg = reader.deserialize(raw, conn.msgtype)
                print(f"[mcap] Loaded frame {frame_index} from {conn.topic}")
                return _pointcloud2_to_xyz(msg)
            seen += 1

        raise RuntimeError(f"Frame {frame_index} not found "
                           f"(only {seen} messages on this topic).")


# ------------------------------------------------------------
# Unified entry point: pick a loader based on WORKFLOW
# ------------------------------------------------------------
# Edit these paths to point at your data:
BIN_PATH  = r"C:/Users/JRepa/OneDrive - Delft University of Technology/Documenten/02. TU Delft/2025-2026/BEP/Data/Pointclouds curvature/pointcloud.bin"
MCAP_PATH = r"C:\rosbag_0.mcap"
def _timestamp_to_index(path, topic=None, target_sec=0.0):
    """Scan the MCAP and return the frame index closest to target_sec."""
    from rosbags.highlevel import AnyReader
    from pathlib import Path

    with AnyReader([Path(path)]) as reader:
        pc2_conns = [c for c in reader.connections
                     if c.msgtype == "sensor_msgs/msg/PointCloud2"]
        if topic is None:
            chosen = pc2_conns[0]
        else:
            chosen = next(c for c in pc2_conns if c.topic == topic)

        best_idx = 0
        best_diff = float("inf")

        for i, (conn, ts, raw) in enumerate(reader.messages(connections=[chosen])):
            diff = abs(ts / 1e9 - target_sec)
            if diff < best_diff:
                best_diff = diff
                best_idx = i

    print(f"[mcap] Timestamp {target_sec:.3f} -> frame {best_idx} "
          f"(off by {best_diff:.4f} s)")
    return best_idx


def load_frame():
    """Dispatch to the correct loader based on WORKFLOW."""
    if WORKFLOW == "bin":
        return load_nuscenes_bin(BIN_PATH)
    elif WORKFLOW == "mcap":
        # Decide which frame to load
        if MCAP_FRAME_TIMESTAMP is not None:
            frame_idx = _timestamp_to_index(MCAP_PATH, MCAP_TOPIC, MCAP_FRAME_TIMESTAMP)
        elif MCAP_FRAME_INDEX is not None:
            frame_idx = MCAP_FRAME_INDEX
        else:
            raise ValueError("Set either MCAP_FRAME_INDEX or MCAP_FRAME_TIMESTAMP in cell 1.")
        
        return load_mcap_frame(MCAP_PATH, topic=MCAP_TOPIC, frame_index=frame_idx)
    else:
        raise ValueError(f"Unknown WORKFLOW '{WORKFLOW}'. Use 'bin' or 'mcap'.")


# Tip: to see which topics are in an MCAP file, run:
# load_mcap_frame(MCAP_PATH, list_topics=True)


points = load_frame()
print(f"Workflow:  {WORKFLOW}")
print(f"Loaded     {len(points):,} points")
print(f"X range:   [{points[:,0].min():.1f}, {points[:,0].max():.1f}] m")
print(f"Y range:   [{points[:,1].min():.1f}, {points[:,1].max():.1f}] m")
print(f"Z range:   [{points[:,2].min():.1f}, {points[:,2].max():.1f}] m")


## 2. Quick look of original pointcloud

Sanity check the data before doing anything clever.

In [ ]:
def bev_plot(points, title="BEV", lim=40, c=None, cmap="viridis",
             cbar_label=None, ax=None, s=1):
    """Top-down scatter of a point cloud. If `c` is None, colour by height."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 7))
    if c is None:
        c = points[:, 2]
    sc = ax.scatter(points[:, 0], points[:, 1], c=c, s=s, cmap=cmap)
    ax.set_xlabel("X forward [m]")
    ax.set_ylabel("Y left [m]")
    ax.set_title(title)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.grid(True, lw=0.3)
    plt.colorbar(sc, ax=ax, label=cbar_label or "Z [m]", shrink=0.8)
    return ax

# --- Side view (X-Z): shows slopes, hills, and the ground profile ---
def side_plot(points, title="Side view", lim=40, c=None, cmap="viridis",
              cbar_label=None, ax=None, s=1):
    """Side projection: X (forward) vs Z (height)."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 4))
    if c is None:
        c = points[:, 1]  # colour by Y (left-right) to add depth
    sc = ax.scatter(points[:, 0], points[:, 2], c=c, s=s, cmap=cmap)
    ax.set_xlabel("X forward [m]")
    ax.set_ylabel("Z height [m]")
    ax.set_title(title)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-5, 10)
    ax.set_aspect("equal")
    ax.grid(True, lw=0.3)
    plt.colorbar(sc, ax=ax, label=cbar_label or "Y [m]", shrink=0.8)
    return ax


# --- Front view (Y-Z): shows the road cross-section ---
def front_plot(points, title="Front view", lim=40, c=None, cmap="viridis",
               cbar_label=None, ax=None, s=1):
    """Front projection: Y (left-right) vs Z (height)."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 4))
    if c is None:
        c = points[:, 0]  # colour by X (distance ahead)
    sc = ax.scatter(points[:, 1], points[:, 2], c=c, s=s, cmap=cmap)
    ax.set_xlabel("Y left [m]")
    ax.set_ylabel("Z height [m]")
    ax.set_title(title)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-5, 10)
    ax.set_aspect("equal")
    ax.grid(True, lw=0.3)
    plt.colorbar(sc, ax=ax, label=cbar_label or "X [m]", shrink=0.8)
    return ax


# --- Show all three views together ---
fig, axes = plt.subplots(3, 1, figsize=(12, 16))

# BEV (top-down)
bev_plot(points, title="Bird's-eye view — coloured by height", ax=axes[0])

# Side view (X-Z)
side_plot(points, title="Side view — coloured by Y (left-right)", ax=axes[1])

# Front view (Y-Z)
front_plot(points, title="Front view — coloured by X (forward)", ax=axes[2])

plt.tight_layout()
plt.show()


## 3. Stage 1 — Ground filter (Patchwork++)

We use the authors' official C++ implementation via the Python binding `pypatchworkpp`. Prebuilt Windows wheels exist for Python 3.8 – 3.12 so `pip install pypatchworkpp` Just Works in most environments.

### What the library actually does — a one-paragraph recap

Patchwork++ splits the ground around the sensor into **concentric zones**, each zone into **angular wedges** (a CZM — Concentric Zone Model). Inside each wedge it runs **R-GPF**: sort points by height, take the lowest slice as "seeds", fit a plane through the seeds with PCA, reject wedges whose plane is too vertical (walls) or whose seeds are too spread out in z (bushes), and label everything within a distance threshold of the plane as ground. The `++` version adds Temporal Ground Revert (fixes under-segmentation using prior frames) and Reflected-Noise Removal. See the paper's Section 2.2 and Lim et al. 2021 / Lee et al. 2022 for the originals.

> **Analogy.** Each wedge is a small patch of parking lot. You stand in it, squint at the ~20 lowest points, lay a ruler across them, and mark every other point within an inch of the ruler as "floor". Everything else = obstacle.

### Sensor-height matters — already configured above

Patchwork++'s defaults assume a SemanticKITTI mount (~1.70 m). Values for other sensors live in `SENSOR_CONFIG` at the top of the notebook. **When you switch to your own LiDAR test rig, change the values there — not here.** The cell below rebuilds the Patchwork++ object from that dict every time it's executed, so re-running cell 2 and cell 8 is enough to apply a new configuration.

### A from-scratch reference implementation is included at the end

The "Appendix A" cell near the bottom contains a minimal pure-Python version (~60 lines) of the same CZM + R-GPF idea. Use it for:

- **Your supervisor meeting** — walking through each threshold line by line.
- **Your defence slides** — proving you understand the method, not just that you called a library.
- **Python 3.13** — where no `pypatchworkpp` wheel exists yet.
- **Sanity checking** — if the C++ result ever looks wrong.

In [ ]:
# ---------------------------------------------------------------
# PCA helper used by BOTH Stage 1 fallback (Appendix A) and Stage 2.
# Defined here so it's available regardless of which ground filter you use.
# ---------------------------------------------------------------
def _pca_plane(pts):
    """Fit a plane through `pts` with PCA.

    Returns
    -------
    n        : (3,) unit normal, oriented so n_z >= 0
    c        : (3,) centroid
    eigvals  : (3,) eigenvalues, sorted descending (lam1 >= lam2 >= lam3)
    """
    c = pts.mean(axis=0)
    cov = np.cov((pts - c).T)
    w, V = np.linalg.eigh(cov)          # ascending eigenvalues
    eigvals = w[::-1]                   # descending
    n = V[:, 0]                         # eigenvector for smallest eigenvalue
    if n[2] < 0:                        # force normal to point up
        n = -n
    return n, c, eigvals


# ---------------------------------------------------------------
# Build the Patchwork++ object from SENSOR_CONFIG.
# Re-run this cell after changing SENSOR_CONFIG to apply new values.
# ---------------------------------------------------------------
def build_patchworkpp(config):
    """Construct a configured Patchwork++ instance from a SENSOR_CONFIG dict.

    Other tuning knobs on `pypatchworkpp.Parameters` that you may want later:
        num_zones, num_rings_each_zone, num_sectors_each_zone
        th_seeds       -- seed-selection height tolerance
        th_dist        -- distance threshold for ground inliers
        uprightness_thr-- reject planes whose normal tilts too much
        RNR_ver_angle_thr, RNR_intensity_thr -- reflected-noise removal knobs
    They all default to sensible values for automotive LiDAR.
    """
    params = pypatchworkpp.Parameters()
    params.sensor_height = float(config["sensor_height"])
    params.min_range     = float(config["min_range"])
    params.max_range     = float(config["max_range"])
    params.verbose       = bool(config.get("verbose", False))
    return pypatchworkpp.patchworkpp(params)


PatchworkPP = build_patchworkpp(SENSOR_CONFIG)
print(f"Patchwork++ configured with sensor_height = {SENSOR_CONFIG['sensor_height']} m, "
      f"range [{SENSOR_CONFIG['min_range']}, {SENSOR_CONFIG['max_range']}] m")


# ---------------------------------------------------------------
# Public API: segment_ground(points) -> (ground, nonground)
# ---------------------------------------------------------------
def segment_ground(points):
    """Run Patchwork++ on an (N, 3) or (N, 4) point cloud.

    Returns
    -------
    ground    : (M, 3) ground points
    nonground : (N - M, 3) obstacle points
    """
    # Patchwork++ expects (N, 4): x, y, z, intensity. Pad with zeros if needed.
    if points.shape[1] == 3:
        pts4 = np.hstack([points, np.zeros((len(points), 1))]).astype(np.float32)
    else:
        pts4 = points[:, :4].astype(np.float32)

    PatchworkPP.estimateGround(pts4)
    ground    = np.asarray(PatchworkPP.getGround())[:, :3]
    nonground = np.asarray(PatchworkPP.getNonground())[:, :3]
    return ground, nonground

In [ ]:
ground, nonground = segment_ground(points)
print(f"Ground:    {len(ground):,} points")
print(f"Obstacles: {len(nonground):,} points")
print(f"Ground fraction: {100 * len(ground) / (len(ground) + len(nonground)):.1f} %")

# Visualize ground vs obstacle
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].scatter(nonground[:, 0], nonground[:, 1], s=1, c="crimson", label="Obstacle")
axes[0].scatter(ground[:, 0],    ground[:, 1],    s=1, c="forestgreen", label="Ground")
axes[0].set_title("Stage 1 result — ground vs obstacle (BEV)")
axes[0].set_xlabel("X [m]"); axes[0].set_ylabel("Y [m]")
axes[0].set_xlim(-40, 40); axes[0].set_ylim(-40, 40)
axes[0].set_aspect("equal"); axes[0].grid(True, lw=0.3); axes[0].legend(markerscale=5)

# Ground only, coloured by height — should now look like terrain
sc = axes[1].scatter(ground[:, 0], ground[:, 1], c=ground[:, 2], s=1, cmap="terrain")
plt.colorbar(sc, ax=axes[1], label="Z [m]", shrink=0.8)
axes[1].set_title("Ground points only — coloured by height")
axes[1].set_xlabel("X [m]"); axes[1].set_ylabel("Y [m]")
axes[1].set_xlim(-40, 40); axes[1].set_ylim(-40, 40)
axes[1].set_aspect("equal"); axes[1].grid(True, lw=0.3)
plt.tight_layout(); plt.show()

# --- Side view: ground vs obstacle ---
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].scatter(nonground[:, 0], nonground[:, 2], s=1, c="crimson", label="Obstacle")
axes[0].scatter(ground[:, 0],    ground[:, 2],    s=1, c="forestgreen", label="Ground")
axes[0].set_title("Stage 1 result — ground vs obstacle (Side view)")
axes[0].set_xlabel("X forward [m]"); axes[0].set_ylabel("Z height [m]")
axes[0].set_xlim(-40, 40); axes[0].set_ylim(-5, 10)
axes[0].set_aspect("equal"); axes[0].grid(True, lw=0.3); axes[0].legend(markerscale=5)

# Ground only, side view coloured by height
sc = axes[1].scatter(ground[:, 0], ground[:, 2], c=ground[:, 2], s=1, cmap="terrain")
plt.colorbar(sc, ax=axes[1], label="Z [m]", shrink=0.8)
axes[1].set_title("Ground points only — side view coloured by height")
axes[1].set_xlabel("X forward [m]"); axes[1].set_ylabel("Z height [m]")
axes[1].set_xlim(-40, 40); axes[1].set_ylim(-5, 10)
axes[1].set_aspect("equal"); axes[1].grid(True, lw=0.3)

plt.tight_layout(); plt.show()


# --- Front view: ground vs obstacle ---
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].scatter(nonground[:, 1], nonground[:, 2], s=1, c="crimson", label="Obstacle")
axes[0].scatter(ground[:, 1],    ground[:, 2],    s=1, c="forestgreen", label="Ground")
axes[0].set_title("Stage 1 result — ground vs obstacle (Front view)")
axes[0].set_xlabel("Y left [m]"); axes[0].set_ylabel("Z height [m]")
axes[0].set_xlim(-40, 40); axes[0].set_ylim(-5, 10)
axes[0].set_aspect("equal"); axes[0].grid(True, lw=0.3); axes[0].legend(markerscale=5)

sc = axes[1].scatter(ground[:, 1], ground[:, 2], c=ground[:, 2], s=1, cmap="terrain")
plt.colorbar(sc, ax=axes[1], label="Z [m]", shrink=0.8)
axes[1].set_title("Ground points only — front view coloured by height")
axes[1].set_xlabel("Y left [m]"); axes[1].set_ylabel("Z height [m]")
axes[1].set_xlim(-40, 40); axes[1].set_ylim(-5, 10)
axes[1].set_aspect("equal"); axes[1].grid(True, lw=0.3)

plt.tight_layout(); plt.show()


# --- 3D interactive: ground vs obstacle ---
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

step_g  = max(1, len(ground) // 20000)
step_ng = max(1, len(nonground) // 10000)

ax.scatter(nonground[::step_ng, 0], nonground[::step_ng, 1], nonground[::step_ng, 2],
           s=0.5, c="crimson", alpha=0.4, label="Obstacle")
ax.scatter(ground[::step_g, 0], ground[::step_g, 1], ground[::step_g, 2],
           s=0.5, c="forestgreen", alpha=0.6, label="Ground")
ax.set_xlabel("X [m]"); ax.set_ylabel("Y [m]"); ax.set_zlabel("Z [m]")
ax.set_title("3D view after Patchwork++ (drag to rotate)")
ax.set_xlim(-40, 40); ax.set_ylim(-40, 40); ax.set_zlim(-5, 10)
ax.legend(markerscale=10)
plt.show()




## 3b. Export the leftover (non-ground) points to a `.bin` file

After Patchwork++ separates ground from obstacles, you sometimes want to hand the **non-ground** cloud off to a downstream tool or collaborator. The cell below saves it back into the same nuScenes-style `.bin` layout (`(N, 5)` float32 with `[x, y, z, intensity, ring]`) so it's a drop-in replacement for any tool that reads nuScenes data -- including this notebook itself, if you re-run cell 1 with `BIN_PATH` pointing at the exported file.

Intensity and ring are zero-padded since Patchwork++ doesn't preserve them (it only returns XYZ). Set `EXPORT_NONGROUND = False` in cell 1 to skip the export step.


In [ ]:
def save_as_nuscenes_bin(points_xyz, path, intensity=0.0, ring=0):
    """Save an (N, 3) XYZ array as a nuScenes-format .bin file.

    Output layout: (N, 5) float32 -> [x, y, z, intensity, ring]
    Matches what `load_nuscenes_bin` reads, so the file round-trips
    cleanly through this notebook.
    """
    n = len(points_xyz)
    arr = np.zeros((n, 5), dtype=np.float32)
    arr[:, :3] = points_xyz.astype(np.float32)
    arr[:, 3]  = intensity
    arr[:, 4]  = ring
    arr.tofile(path)
    print(f"Saved {n:,} points  ->  {path}  ({arr.nbytes / 1024:.1f} KB)")


if EXPORT_GROUND:
    save_as_nuscenes_bin(ground, GROUND_BIN_PATH)

    # Round-trip sanity check: read it back and confirm the count
    reloaded = load_nuscenes_bin(GROUND_BIN_PATH)
    assert len(reloaded) == len(ground), "Round-trip count mismatch!"
    print(f"Round-trip OK -- {len(reloaded):,} points read back from "
          f"'{GROUND_BIN_PATH}'")
else:
    print("EXPORT_GROUND = False, skipping export.")


## 4. Stage 2 — Slope and curvature per grid cell

Now we have clean ground points. Lay a uniform grid over them (simpler to read than a polar grid at this stage) and run PCA per cell. Two numbers come out of one PCA:

### Slope (from the normal vector)
PCA gives us the plane normal `n = (n_x, n_y, n_z)`. The angle it makes with the vertical `z`-axis is the slope:

$$\text{slope} = \arccos(n \cdot \hat z) \quad\text{(in degrees)}$$

### Curvature (from the eigenvalues)
Using paper Eq. (3):

$$\mathcal{C} = \frac{\lambda_3}{\lambda_1 + \lambda_2 + \lambda_3}$$

**Intuition.** $\lambda_1, \lambda_2, \lambda_3$ are the variances of the points along the three principal directions. $\lambda_3$ is the "thickness" of the fit (how much points deviate perpendicular to the plane). Divided by the total variance, it's a dimensionless measure of *how far from planar* the cell is.

- Flat asphalt: $\lambda_3 \approx 0 \Rightarrow \mathcal{C} \approx 0$
- Curved hill crest: $\lambda_3$ grows because points curve away from any single plane $\Rightarrow \mathcal{C}$ grows
- Speed bump inside a cell: $\mathcal{C}$ jumps

**Key distinction for the report.** Slope tells you the *orientation* of the best-fit plane; curvature tells you *how well a single plane fits at all*. A uniformly-tilted ramp has high slope but low curvature. A gently-rolling hill can have low slope but higher curvature.

In [ ]:
def compute_slope_curvature_grid(ground_points,
                                 cell_size=2.0,
                                 min_pts_per_cell=12,
                                 x_range=(-40, 40),
                                 y_range=(-40, 40)):
    """Uniform Cartesian grid over ground points. PCA per cell.

    Returns a dict with the following keys (one entry per populated cell):
        cx, cy  -- centre of the cell [m]
        slope   -- angle between PCA normal and +Z [deg]
        curv    -- lam3 / (lam1 + lam2 + lam3)   (paper Eq. 3)
        n_xy    -- (M, 2) tilt direction of the plane in BEV
    """
    x_edges = np.arange(x_range[0], x_range[1] + cell_size, cell_size)
    y_edges = np.arange(y_range[0], y_range[1] + cell_size, cell_size)

    ix = np.digitize(ground_points[:, 0], x_edges) - 1
    iy = np.digitize(ground_points[:, 1], y_edges) - 1
    valid = (ix >= 0) & (ix < len(x_edges) - 1) & (iy >= 0) & (iy < len(y_edges) - 1)
    ix, iy = ix[valid], iy[valid]
    pts = ground_points[valid]

    z_axis = np.array([0.0, 0.0, 1.0])
    cxs, cys, slopes, curvs, nxys = [], [], [], [], []

    # Bundle (ix, iy) into one integer key so we can group via np.unique — avoids a double loop
    lin = ix * len(y_edges) + iy
    for lin_key in np.unique(lin):
        cell_pts = pts[lin == lin_key]
        if len(cell_pts) < min_pts_per_cell:
            continue
        n, _, eigvals = _pca_plane(cell_pts)
        slope_deg = np.degrees(np.arccos(np.clip(n @ z_axis, -1.0, 1.0)))
        curvature = eigvals[2] / (eigvals.sum() + 1e-12)

        i_cell = lin_key // len(y_edges)
        j_cell = lin_key %  len(y_edges)
        cxs.append(0.5 * (x_edges[i_cell] + x_edges[i_cell + 1]))
        cys.append(0.5 * (y_edges[j_cell] + y_edges[j_cell + 1]))
        slopes.append(slope_deg)
        curvs.append(curvature)
        nxys.append(n[:2])

    return dict(
        cx=np.array(cxs), cy=np.array(cys),
        slope=np.array(slopes), curv=np.array(curvs),
        n_xy=np.array(nxys) if nxys else np.zeros((0, 2)),
    )


def plot_slope_curvature(g, cell_size, suptitle=""):
    """Side-by-side slope and curvature heatmaps. Arrows show tilt direction per cell."""
    if len(g["cx"]) == 0:
        print("No populated cells.")
        return

    fig, axes = plt.subplots(1, 2, figsize=(15, 7))

    # --- slope ---
    vmax = float(np.percentile(g["slope"], 95))
    sc = axes[0].scatter(g["cx"], g["cy"], c=g["slope"], cmap="RdYlGn_r",
                         s=(cell_size * 9) ** 2, marker="s", vmin=0, vmax=vmax)
    for i in range(len(g["cx"])):
        d = g["n_xy"][i]
        m = np.linalg.norm(d)
        if m > 1e-3:
            d = d / m * cell_size * 0.45
            axes[0].annotate(
                "", xy=(g["cx"][i] + d[0], g["cy"][i] + d[1]),
                xytext=(g["cx"][i], g["cy"][i]),
                arrowprops=dict(arrowstyle="->", color="black", lw=0.8),
            )
    axes[0].plot(0, 0, "o", color="dodgerblue", ms=10, label="Sensor")
    axes[0].set_title(f"Slope [deg]   (cell = {cell_size} m)")
    plt.colorbar(sc, ax=axes[0], label="deg", shrink=0.8)

    # --- curvature ---
    vmax_c = float(np.percentile(g["curv"], 95))
    sc2 = axes[1].scatter(g["cx"], g["cy"], c=g["curv"], cmap="magma_r",
                          s=(cell_size * 9) ** 2, marker="s", vmin=0, vmax=vmax_c)
    axes[1].plot(0, 0, "o", color="dodgerblue", ms=10, label="Sensor")
    axes[1].set_title(r"Curvature  $\mathcal{C} = \lambda_3 / \sum\lambda_i$")
    plt.colorbar(sc2, ax=axes[1], label="curvature", shrink=0.8)

    for ax in axes:
        ax.set_xlabel("X [m]"); ax.set_ylabel("Y [m]")
        ax.set_aspect("equal"); ax.grid(True, lw=0.3)
        ax.legend(loc="upper right")
    if suptitle:
        fig.suptitle(suptitle)
    plt.tight_layout()
    plt.show()


CELL = 2.0
grid = compute_slope_curvature_grid(ground, cell_size=CELL)

print(f"Populated cells: {len(grid['cx'])}")
print(f"Slope     -- mean {grid['slope'].mean():5.2f} deg  max {grid['slope'].max():5.2f} deg")
print(f"Curvature -- mean {grid['curv'].mean():.4f}  max {grid['curv'].max():.4f}")

plot_slope_curvature(grid, CELL, suptitle="Stage 2 -- slope & curvature on filtered ground")

## 5. Stage 3 — A whole drive: trajectory + per-frame slope/curvature

Point this at a **folder of `.bin` files** and it will:

1. Run Stages 1–2 on every frame.
2. At each frame, record the slope and curvature of the cells right underneath the sensor (within a small radius around `(0,0)` in the local frame) — that's what the IMU would feel.
3. Optionally transform per-frame sensor positions into a global frame if you give it a poses file.
4. Save a CSV with `[frame, timestamp, x, y, z, slope, curvature]` for direct comparison against your IMU log.

**About poses.** If you have them, provide a CSV with columns `timestamp, x, y, z` (3-D trajectory) or `timestamp, x, y, z, qw, qx, qy, qz` (full pose) — rows in the same order as the sorted `.bin` files. If you don't, the function falls back to using the frame index as the x-axis, which still gives you a time series of slope/curvature that you can cross-correlate with an IMU stream by timestamp.

In [ ]:
def process_sequence(bin_folder,
                     poses_csv=None,
                     cell_size=2.0,
                     ego_radius=3.0,
                     verbose=True):
    """Run Stages 1-2 on every .bin in a folder and log sensor-local slope/curvature.

    Parameters
    ----------
    bin_folder : str
        Folder containing .bin files. Processed in sorted filename order.
    poses_csv  : str or None
        Optional CSV with at least columns [timestamp, x, y, z]. Rows must match
        the sorted .bin files one-for-one. If None, frame index is used.
    cell_size  : float
        Same as Stage 2.
    ego_radius : float
        Radius [m] around (0, 0) in the local frame used to pick the cells that
        represent "what the vehicle is driving over right now".

    Returns
    -------
    dict of np.ndarray keyed by: frame, timestamp, x, y, z, slope, curv
    """
    bin_files = sorted(glob.glob(os.path.join(bin_folder, "*.bin")))
    if not bin_files:
        raise FileNotFoundError(f"No .bin files in {bin_folder}")

    poses = None
    if poses_csv is not None:
        poses = np.genfromtxt(poses_csv, delimiter=",", names=True)
        if len(poses) != len(bin_files):
            print(f"[warn] {len(poses)} poses vs {len(bin_files)} bins -- "
                  "truncating to the shorter one.")
        L = min(len(poses), len(bin_files))
        bin_files = bin_files[:L]

    out = {k: [] for k in ("frame", "timestamp", "x", "y", "z", "slope", "curv")}

    for fi, bf in enumerate(bin_files):
        pc = load_nuscenes_bin(bf)
        g, _ = segment_ground(pc)
        grid = compute_slope_curvature_grid(g, cell_size=cell_size)

        # Average slope/curv of cells within ego_radius of the sensor
        if len(grid["cx"]) == 0:
            slope_here = curv_here = np.nan
        else:
            r = np.sqrt(grid["cx"] ** 2 + grid["cy"] ** 2)
            near = r <= ego_radius
            if near.any():
                slope_here = float(np.nanmean(grid["slope"][near]))
                curv_here  = float(np.nanmean(grid["curv"][near]))
            else:
                slope_here = curv_here = np.nan

        if poses is not None:
            out["timestamp"].append(float(poses["timestamp"][fi]))
            out["x"].append(float(poses["x"][fi]))
            out["y"].append(float(poses["y"][fi]))
            out["z"].append(float(poses["z"][fi]))
        else:
            out["timestamp"].append(float(fi))
            out["x"].append(float(fi))   # placeholder: frame index on x-axis
            out["y"].append(0.0)
            out["z"].append(0.0)

        out["frame"].append(fi)
        out["slope"].append(slope_here)
        out["curv"].append(curv_here)

        if verbose and fi % 10 == 0:
            print(f"[{fi + 1}/{len(bin_files)}] {os.path.basename(bf)}  "
                  f"slope={slope_here:.2f} deg  curv={curv_here:.4f}")

    return {k: np.array(v) for k, v in out.items()}


# -------- usage example --------
# Point this at a folder containing a sequence of .bin files.
SEQ_FOLDER = r"C:/Users/JRepa/OneDrive - Delft University of Technology/Documenten/02. TU Delft/2025-2026/BEP/Data/Pointclouds curvature/sequence"
POSES_CSV  = None   # or path to "poses.csv" with header: timestamp,x,y,z

# Uncomment to run on a full sequence:
# traj = process_sequence(SEQ_FOLDER, poses_csv=POSES_CSV)
# print("Done. Frames processed:", len(traj['frame']))

In [ ]:
def plot_trajectory(traj):
    """Two plots:
      (a) trajectory in XY coloured by slope
      (b) slope & curvature vs frame/time
    """
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    valid = ~np.isnan(traj["slope"])
    if valid.any():
        sc = axes[0].scatter(traj["x"][valid], traj["y"][valid],
                             c=traj["slope"][valid], cmap="RdYlGn_r",
                             s=20, vmin=0,
                             vmax=np.nanpercentile(traj["slope"], 95))
        plt.colorbar(sc, ax=axes[0], label="Slope [deg]", shrink=0.8)
    axes[0].plot(traj["x"], traj["y"], "-", color="gray", lw=0.6, zorder=0)
    axes[0].set_xlabel("X [m]"); axes[0].set_ylabel("Y [m]")
    axes[0].set_title("Trajectory coloured by local slope")
    axes[0].set_aspect("equal"); axes[0].grid(True, lw=0.3)

    # time series -- slope and curvature on a twin y-axis
    ax2 = axes[1]
    ax2.plot(traj["frame"], traj["slope"], color="tab:blue", label="slope [deg]")
    ax2.set_xlabel("Frame"); ax2.set_ylabel("Slope [deg]", color="tab:blue")
    ax2.tick_params(axis="y", labelcolor="tab:blue")
    ax2b = ax2.twinx()
    ax2b.plot(traj["frame"], traj["curv"], color="tab:red", label="curvature")
    ax2b.set_ylabel("Curvature", color="tab:red")
    ax2b.tick_params(axis="y", labelcolor="tab:red")
    ax2.set_title("Per-frame slope & curvature")
    ax2.grid(True, lw=0.3)

    plt.tight_layout()
    plt.show()


def save_trajectory_csv(traj, path="lidar_slope_curvature.csv"):
    """Write per-frame result as CSV -- ready to merge with an IMU log by timestamp."""
    header = "frame,timestamp,x,y,z,slope_deg,curvature"
    stacked = np.column_stack([traj["frame"], traj["timestamp"],
                               traj["x"], traj["y"], traj["z"],
                               traj["slope"], traj["curv"]])
    np.savetxt(path, stacked, delimiter=",", header=header, comments="",
               fmt=["%d", "%.6f", "%.3f", "%.3f", "%.3f", "%.3f", "%.6f"])
    print(f"Saved: {path}")


# -------- usage example --------
# plot_trajectory(traj)
# save_trajectory_csv(traj, "lidar_slope_curvature.csv")

## Notes you can lift into the BEP report

**Design decisions (and *why*):**

| Choice | Alternative considered | Reason for the chosen option |
|--------|-----------------------|------------------------------|
| Patchwork++ for Stage 1 | Single RANSAC plane; Ray Ground Filter; home-grown CZM | Authors' own implementation, peer-reviewed, proven on SemanticKITTI (F1 ≈ 83%), and robust to hills / ramps thanks to the concentric-zone model. |
| PCA for local planes in Stage 2 | RANSAC | Deterministic, uses every point, closed-form. After Stage 1 the cell is already mostly ground, so consensus-from-samples buys nothing and adds randomness to the slope field. |
| Uniform Cartesian grid for Stage 2 | Reuse CZM | Easier to read in plots and to cross-reference with the vehicle's path. Obstacle removal is already done upstream by Patchwork++. |
| $\lambda_3/\sum\lambda_i$ as curvature (Eq. 3) | Mean curvature from a quadric fit | Cheap, dimensionless, co-computed with the normal in the same PCA step — no extra pass. |
| `SENSOR_CONFIG` dict at the top | Hard-coded numbers scattered through Stage 1 | One place to change when moving between nuScenes, KITTI, and your own test rig — prevents silent mis-configuration on a new sensor. |

**Validation experiments worth running:**

- **Scale sweep.** Vary `cell_size` over `{1, 2, 3, 5}` m and plot how the slope/curvature distributions change. Shows the spatial-scale trade-off that any terrain estimator has to defend.
- **IMU cross-check.** Cross-correlate `slope(t)` (from this pipeline) against IMU pitch, and tilt-direction against IMU roll. Report the lag of the peak correlation (good pipeline ⇒ lag ≈ 0) and the RMSE in the overlapping band.
- **Flat-ground baseline.** Run the whole pipeline on a scene you *know* is flat (empty parking lot). Slope and curvature should be near zero everywhere — if they aren't, you've found a noise floor to report.
- **Sensor-height sensitivity.** Sweep `SENSOR_CONFIG["sensor_height"]` by ±10 cm and record the change in ground-point fraction. Useful for quantifying how carefully the LiDAR needs to be mounted on your test rig.
- **Library vs reference implementation.** Run Stage 1 with `segment_ground` (Patchwork++) and with `segment_ground_fallback` (Appendix A) on the same frames; plot ground-fraction per frame. Agreement at ~same level across the scene is strong evidence you understand what the library does.

**What's *not* in this pipeline (and why that's fine).** The TE-NeXt sparse neural network from Santo et al. is deliberately skipped. Its role in the paper is to rescue sparse far-range sectors where geometric methods run out of points. For your IMU-comparison work the vehicle only "feels" the terrain under and immediately around it, so the sparse far range doesn't enter the measurement, and the geometric core is sufficient and keeps the method interpretable for your defence.

---

## Appendix A — What Patchwork++ actually does, in ~60 lines of Python

The cell below is a from-scratch reference implementation of the core Patchwork++ idea (CZM + R-GPF). It omits the advanced extras the real library ships with (Temporal Ground Revert, Reflected Noise Removal, RVPF) so it is **strictly weaker** than `pypatchworkpp` — but it's a useful thing to have in your back pocket because:

1. **Every threshold is explicit and named.** In a supervisor meeting you can point at `min_normal_z = 0.8` and say "this rejects planes whose normal deviates more than 37° from vertical — so walls."
2. **You understand the algorithm, not just the API.** Good for the defence Q&A.
3. **It runs on Python 3.13** and anywhere else where the compiled wheel isn't available.
4. **You can cross-check the C++ library.** If `pypatchworkpp` ever gives you something surprising, rerun Stage 1 through `czm_ground_filter` below and compare ground fractions.

It exposes the same `(ground, nonground)` interface as `segment_ground`, so you can swap it in by assigning `segment_ground = segment_ground_fallback` just before you run Stages 2 and 3.

In [ ]:
# -----------------------------------------------------------
# Concentric Zone Model -- definition
# -----------------------------------------------------------
# Each row: (r_min, r_max, n_rings, n_sectors)
# Further zones get more sectors because they cover more arc length.
CZM_ZONES = [
    # r_min  r_max  n_rings  n_sectors
    (   0.0,   4.0,       2,        16),
    (   4.0,  12.0,       3,        32),
    (  12.0,  25.0,       3,        54),
    (  25.0,  50.0,       2,        32),
]


def czm_ground_filter(points,
                      zones=CZM_ZONES,
                      seed_band=0.5,          # [m] thickness of the lowest seed band
                      min_seeds=6,            # need this many seeds to fit a plane
                      max_seed_sigma_z=0.35,  # [m] reject bush-like noisy seeds
                      min_normal_z=0.8,       # reject near-vertical planes (walls)
                      k_sigma=3.0,            # normalised distance threshold (paper Eq. 5)
                      absolute_max_dist=0.4): # [m] hard cap on point-to-plane distance
    """Patchwork++-style ground filter: CZM + R-GPF per sector. Pure Python.

    Returns a boolean mask of the same length as `points` (True = ground).
    """
    x, y, z = points[:, 0], points[:, 1], points[:, 2]
    r = np.sqrt(x * x + y * y)
    theta = np.arctan2(y, x)

    ground_mask = np.zeros(len(points), dtype=bool)

    for r0, r1, n_rings, n_sectors in zones:
        ring_edges = np.linspace(r0, r1, n_rings + 1)
        sec_edges = np.linspace(-np.pi, np.pi, n_sectors + 1)

        for ri in range(n_rings):
            in_ring = (r >= ring_edges[ri]) & (r < ring_edges[ri + 1])
            for si in range(n_sectors):
                in_sec = (theta >= sec_edges[si]) & (theta < sec_edges[si + 1])
                sector_idx = np.where(in_ring & in_sec)[0]
                if len(sector_idx) < min_seeds:
                    continue

                sec_pts = points[sector_idx]

                # 1. Seed selection: the lowest band of points in this wedge
                z_min = sec_pts[:, 2].min()
                seeds = sec_pts[sec_pts[:, 2] < z_min + seed_band]
                if len(seeds) < min_seeds:
                    continue

                sigma_z = seeds[:, 2].std()
                if sigma_z > max_seed_sigma_z:            # bush or a steep slope
                    continue

                # 2. PCA plane through the seeds
                n, c, _ = _pca_plane(seeds)
                if n[2] < min_normal_z:                   # plane too vertical => wall
                    continue

                # 3. Label inliers in this wedge
                d_abs = np.abs((sec_pts - c) @ n)
                d_norm = d_abs / (sigma_z + 1e-6)         # paper Eq. 5
                inlier = (d_norm <= k_sigma) & (d_abs <= absolute_max_dist)
                ground_mask[sector_idx[inlier]] = True

    return ground_mask


def segment_ground_fallback(points):
    """Same interface as `segment_ground`, but runs the pure-Python filter above.

    Usage: if you want to use the fallback instead of pypatchworkpp, do:
        segment_ground = segment_ground_fallback
    ...before calling Stage 2 / Stage 3.
    """
    mask = czm_ground_filter(points[:, :3])
    return points[mask, :3], points[~mask, :3]


# -------- usage example --------
# g_py, ng_py = segment_ground_fallback(points)
# print(f"Python fallback ground fraction: "
#       f"{100 * len(g_py) / (len(g_py) + len(ng_py)):.1f} %")
# (Compare with the pypatchworkpp result you printed earlier.)